## 1. Imports et configuration

On installe / importe toutes les dépendances nécessaires pour le pipeline complet.

In [ ]:
# Installation des dépendances (décommenter si nécessaire)
# !pip install scikit-learn sentence-transformers matplotlib seaborn

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Configuration de l'affichage
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.max_colwidth', 80)

print("✅ Imports réussis")

## 2. Chargement du dataset

Le fichier `noisecleaner_dataset_mvp_fixed.json` contient un tableau JSON d'articles, chacun ayant :
- `article_id` : identifiant unique de l'article
- `segments` : liste de chaînes de texte (segments de la page)
- `labels` : liste d'entiers `0` (bruit) ou `1` (éditorial)

In [ ]:
DATASET_PATH = "noisecleaner_dataset_mvp_fixed.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"✅ Dataset chargé avec succès")
print(f"   Nombre d'articles : {len(dataset)}")
print(f"   Nombre total de segments : {sum(len(a['segments']) for a in dataset)}")

## 3. Validation stricte du schéma

Avant toute analyse, on vérifie que **chaque article** respecte le schéma MVP :
1. `article_id` est une chaîne non vide et unique
2. `segments` est une liste de chaînes non vides
3. `labels` est une liste d'entiers binaires (`0` ou `1`)
4. `len(segments) == len(labels)`
5. Aucun champ supplémentaire n'est présent

In [ ]:
REQUIRED_KEYS = {"article_id", "segments", "labels"}
errors = []
article_ids = []

for i, article in enumerate(dataset):
    prefix = f"Article[{i}]"
    
    # 1. Vérifier les clés
    keys = set(article.keys())
    if keys != REQUIRED_KEYS:
        extra = keys - REQUIRED_KEYS
        missing = REQUIRED_KEYS - keys
        if extra:
            errors.append(f"{prefix}: Clés en trop → {extra}")
        if missing:
            errors.append(f"{prefix}: Clés manquantes → {missing}")
        continue
    
    aid = article["article_id"]
    segs = article["segments"]
    labs = article["labels"]
    
    # 2. article_id
    if not isinstance(aid, str) or len(aid.strip()) == 0:
        errors.append(f"{prefix}: article_id invalide → '{aid}'")
    article_ids.append(aid)
    
    # 3. segments = liste de strings non vides
    if not isinstance(segs, list):
        errors.append(f"{prefix}: segments n'est pas une liste")
    else:
        for j, seg in enumerate(segs):
            if not isinstance(seg, str) or len(seg.strip()) == 0:
                errors.append(f"{prefix}.segments[{j}]: segment vide ou non-string")
    
    # 4. labels = liste de 0/1
    if not isinstance(labs, list):
        errors.append(f"{prefix}: labels n'est pas une liste")
    else:
        for j, lab in enumerate(labs):
            if lab not in (0, 1):
                errors.append(f"{prefix}.labels[{j}]: label invalide → {lab}")
    
    # 5. Longueurs cohérentes
    if isinstance(segs, list) and isinstance(labs, list):
        if len(segs) != len(labs):
            errors.append(f"{prefix}: len(segments)={len(segs)} ≠ len(labels)={len(labs)}")

# 6. Unicité des article_id
duplicates = [aid for aid, count in Counter(article_ids).items() if count > 1]
if duplicates:
    errors.append(f"article_id en double : {duplicates}")

# Résultat
if errors:
    print("❌ ÉCHEC de la validation :")
    for e in errors:
        print(f"   • {e}")
else:
    print("✅ Validation réussie — Toutes les vérifications passent :")
    print(f"   • {len(dataset)} articles avec article_id unique")
    print(f"   • Toutes les paires (segments, labels) sont alignées")
    print(f"   • Labels strictement binaires (0 ou 1)")
    print(f"   • Aucun champ supplémentaire détecté")

## 4. Construction du DataFrame

On « aplatit » les articles en un DataFrame à une ligne par segment, ce qui facilite l'analyse exploratoire et l'entraînement du modèle.

| Colonne | Description |
|:---|:---|
| `article_id` | Identifiant de l'article source |
| `segment` | Texte brut du segment |
| `label` | 0 = bruit, 1 = éditorial |
| `seg_len` | Longueur en caractères du segment |
| `language` | Langue déduite de l'article_id |

In [ ]:
rows = []
for article in dataset:
    aid = article["article_id"]
    for seg, lab in zip(article["segments"], article["labels"]):
        rows.append({
            "article_id": aid,
            "segment": seg,
            "label": lab,
            "seg_len": len(seg)
        })

df = pd.DataFrame(rows)

# Déduire la langue à partir de l'article_id (fr_, ar_, en_)
def extract_language(aid):
    prefix = aid.split("_")[0]
    mapping = {"fr": "Français", "ar": "Arabe", "en": "Anglais"}
    return mapping.get(prefix, "Inconnu")

df["language"] = df["article_id"].apply(extract_language)

print(f"✅ DataFrame créé : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print()
df.head(10)

## 5. Analyse exploratoire (EDA)

### 5.1 Distribution des labels

On vérifie l'équilibre entre les classes *éditorial* et *bruit*.

In [ ]:
label_counts = df["label"].value_counts()
label_pct = df["label"].value_counts(normalize=True) * 100

print("Distribution des labels :")
print(f"   Bruit (0)      : {label_counts[0]:>4} segments ({label_pct[0]:.1f}%)")
print(f"   Éditorial (1)  : {label_counts[1]:>4} segments ({label_pct[1]:.1f}%)")
print(f"   Ratio bruit/éditorial : {label_counts[0]/label_counts[1]:.2f}:1")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = ["#e74c3c", "#2ecc71"]
axes[0].bar(["Bruit (0)", "Éditorial (1)"], label_counts.values, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Nombre de segments")
axes[0].set_title("Distribution des labels")
for i, (v, p) in enumerate(zip(label_counts.values, label_pct.values)):
    axes[0].text(i, v + 5, f"{v} ({p:.1f}%)", ha="center", fontweight="bold")

# Pie chart
axes[1].pie(label_counts.values, labels=["Bruit (0)", "Éditorial (1)"], 
            colors=colors, autopct="%1.1f%%", startangle=90,
            explode=(0.05, 0.05), shadow=True)
axes[1].set_title("Proportion des classes")

plt.tight_layout()
plt.show()

### 5.2 Distribution par langue

Le dataset est **trilingue** (Français, Arabe, Anglais). On vérifie l'équilibre.

In [ ]:
lang_stats = df.groupby("language").agg(
    articles=("article_id", "nunique"),
    segments=("segment", "count"),
    editorial=("label", "sum"),
    noise=("label", lambda x: (x == 0).sum())
).reset_index()

lang_stats["pct_editorial"] = (lang_stats["editorial"] / lang_stats["segments"] * 100).round(1)

print("Distribution par langue :")
print(lang_stats.to_string(index=False))
print()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Segments par langue
lang_colors = {"Français": "#3498db", "Arabe": "#e67e22", "Anglais": "#9b59b6"}
bars = axes[0].bar(lang_stats["language"], lang_stats["segments"], 
                   color=[lang_colors[l] for l in lang_stats["language"]], 
                   edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Nombre de segments")
axes[0].set_title("Segments par langue")
for bar, val in zip(bars, lang_stats["segments"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, 
                 str(val), ha="center", fontweight="bold")

# Stacked bar: editorial vs noise par langue
x = range(len(lang_stats))
axes[1].bar(x, lang_stats["editorial"], label="Éditorial (1)", color="#2ecc71", edgecolor="black", linewidth=0.5)
axes[1].bar(x, lang_stats["noise"], bottom=lang_stats["editorial"], label="Bruit (0)", color="#e74c3c", edgecolor="black", linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(lang_stats["language"])
axes[1].set_ylabel("Nombre de segments")
axes[1].set_title("Répartition éditorial/bruit par langue")
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.3 Distribution de la longueur des segments

On s'attend à ce que les segments éditoriaux soient **plus longs** (paragraphes) et les segments de bruit **plus courts** (menus, boutons, liens).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme par label
for label, color, name in [(0, "#e74c3c", "Bruit"), (1, "#2ecc71", "Éditorial")]:
    subset = df[df["label"] == label]["seg_len"]
    axes[0].hist(subset, bins=50, alpha=0.6, color=color, label=f"{name} (μ={subset.mean():.0f})")

axes[0].set_xlabel("Longueur du segment (caractères)")
axes[0].set_ylabel("Fréquence")
axes[0].set_title("Distribution de la longueur par classe")
axes[0].legend()
axes[0].axvline(x=40, color="gray", linestyle="--", alpha=0.5, label="Seuil 40 chars")

# Boxplot par label
df.boxplot(column="seg_len", by="label", ax=axes[1])
axes[1].set_xlabel("Label (0=Bruit, 1=Éditorial)")
axes[1].set_ylabel("Longueur (caractères)")
axes[1].set_title("Boxplot de la longueur par classe")
plt.suptitle("")  # Supprimer le titre auto

plt.tight_layout()
plt.show()

# Statistiques descriptives
print("Statistiques de longueur par classe :")
print(df.groupby("label")["seg_len"].describe().round(1).to_string())

### 5.4 Vue par article

Visualisation du nombre de segments éditoriaux vs bruit pour chaque article.

In [ ]:
article_stats = df.groupby("article_id").agg(
    total=("label", "count"),
    editorial=("label", "sum"),
    noise=("label", lambda x: (x == 0).sum())
).reset_index()
article_stats["pct_editorial"] = (article_stats["editorial"] / article_stats["total"] * 100).round(1)
article_stats = article_stats.sort_values("article_id")

fig, ax = plt.subplots(figsize=(14, 6))
x = range(len(article_stats))
ax.barh(x, article_stats["editorial"], color="#2ecc71", label="Éditorial (1)", edgecolor="black", linewidth=0.3)
ax.barh(x, article_stats["noise"], left=article_stats["editorial"], color="#e74c3c", label="Bruit (0)", edgecolor="black", linewidth=0.3)
ax.set_yticks(x)
ax.set_yticklabels(article_stats["article_id"], fontsize=8)
ax.set_xlabel("Nombre de segments")
ax.set_title("Répartition éditorial/bruit par article")
ax.legend(loc="lower right")

# Annotations
for i, row in enumerate(article_stats.itertuples()):
    ax.text(row.total + 0.5, i, f"{row.pct_editorial}%", va="center", fontsize=7, color="gray")

plt.tight_layout()
plt.show()

### 5.5 Exemples de segments

Quelques exemples concrets de segments éditoriaux et de bruit pour comprendre la tâche.

In [ ]:
print("=" * 80)
print("📰 SEGMENTS ÉDITORIAUX (label=1) — 5 exemples aléatoires")
print("=" * 80)
for _, row in df[df["label"] == 1].sample(5, random_state=42).iterrows():
    print(f"  [{row['article_id']}] {row['segment'][:120]}...")
    print()

print("=" * 80)
print("🗑️ SEGMENTS DE BRUIT (label=0) — 5 exemples aléatoires")
print("=" * 80)
for _, row in df[df["label"] == 0].sample(5, random_state=42).iterrows():
    print(f"  [{row['article_id']}] {row['segment'][:120]}")
    print()

## 6. Préparation des données pour l'entraînement

### 6.1 Vectorisation avec Sentence-Transformers

On utilise **`paraphrase-multilingual-MiniLM-L12-v2`** qui produit des embeddings de 384 dimensions pour n'importe quelle langue (FR, AR, EN).

> ⚡ Le modèle sera téléchargé automatiquement la première fois (~120 MB). Les exécutions suivantes seront rapides grâce au cache.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"⏳ Chargement du modèle : {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)

print(f"✅ Modèle chargé — Dimension des embeddings : {model.get_sentence_embedding_dimension()}")

In [ ]:
print("⏳ Vectorisation de tous les segments...")
segments_text = df["segment"].tolist()
X = model.encode(segments_text, show_progress_bar=True, batch_size=64)
y = df["label"].values

print(f"✅ Vectorisation terminée")
print(f"   X.shape = {X.shape}  (segments × dimensions)")
print(f"   y.shape = {y.shape}  (labels)")

### 6.2 Séparation train/test

> ⚠️ **IMPORTANT** : On utilise `GroupShuffleSplit` pour séparer les données au niveau de l'**article** (pas du segment). Cela empêche la fuite de données (*data leakage*), car un même texte peut apparaître comme titre dans un article (label=1) et comme lien dans un autre (label=0).

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["article_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# Articles dans chaque split
train_articles = set(groups[train_idx])
test_articles = set(groups[test_idx])

print(f"✅ Split train/test au niveau article")
print(f"   Train : {len(X_train)} segments ({len(train_articles)} articles)")
print(f"   Test  : {len(X_test)} segments ({len(test_articles)} articles)")
print()
print(f"   Articles train : {sorted(train_articles)}")
print(f"   Articles test  : {sorted(test_articles)}")
print()

# Vérification : aucun article en commun
overlap = train_articles & test_articles
if overlap:
    print(f"❌ FUITE : Articles communs détectés → {overlap}")
else:
    print(f"✅ Aucune fuite — 0 article en commun entre train et test")

## 7. Entraînement du modèle — Logistic Regression

On entraîne un classificateur **Logistic Regression** sur les embeddings MiniLM.

Paramètres clés :
- `class_weight='balanced'` : compense le déséquilibre 70:30 bruit/éditorial
- `max_iter=1000` : assure la convergence
- `C=1.0` : régularisation par défaut (L2)

> 💡 Ce modèle linéaire est adapté à un petit dataset (571 segments). Un réseau de neurones risquerait le surapprentissage.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    C=1.0,
    random_state=42,
    solver="lbfgs"
)

print("⏳ Entraînement en cours...")
clf.fit(X_train, y_train)
print(f"✅ Entraînement terminé")
print(f"   Convergé en {clf.n_iter_[0]} itérations")
print(f"   Accuracy sur train : {clf.score(X_train, y_train):.4f}")

## 8. Évaluation du modèle

### 8.1 Métriques complètes

On évalue le modèle avec :
- **Accuracy** : proportion de prédictions correctes
- **Precision** : parmi les segments prédits éditoriaux, combien le sont vraiment ?
- **Recall** : parmi les vrais segments éditoriaux, combien sont détectés ?
- **F1-score** : moyenne harmonique de la precision et du recall
- **Matrice de confusion** : visualisation des erreurs

In [ ]:
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

y_pred = clf.predict(X_test)

print("=" * 60)
print("📊 RAPPORT DE CLASSIFICATION")
print("=" * 60)
print()
print(classification_report(
    y_test, y_pred, 
    target_names=["Bruit (0)", "Éditorial (1)"],
    digits=4
))
print("=" * 60)
print(f"Accuracy globale : {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro         : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1 weighted      : {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("=" * 60)

### 8.2 Matrice de confusion

La matrice de confusion montre les 4 cas possibles :
- **Vrai Négatif (VN)** : bruit correctement identifié comme bruit
- **Faux Positif (FP)** : bruit incorrectement classé comme éditorial
- **Faux Négatif (FN)** : éditorial incorrectement classé comme bruit
- **Vrai Positif (VP)** : éditorial correctement identifié comme éditorial

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Bruit (0)", "Éditorial (1)"],
            yticklabels=["Bruit (0)", "Éditorial (1)"],
            linewidths=0.5, linecolor="black",
            annot_kws={"size": 16, "fontweight": "bold"})
ax.set_xlabel("Prédiction", fontsize=12)
ax.set_ylabel("Réalité", fontsize=12)
ax.set_title("Matrice de Confusion", fontsize=14, fontweight="bold")

# Annotations dans les coins
total = cm.sum()
for i in range(2):
    for j in range(2):
        pct = cm[i, j] / total * 100
        labels = [["VN", "FP"], ["FN", "VP"]]
        ax.text(j + 0.5, i + 0.75, f"({labels[i][j]} — {pct:.1f}%)", 
                ha="center", va="center", fontsize=9, color="gray")

plt.tight_layout()
plt.show()

### 8.3 Performance par langue

Le modèle doit fonctionner de manière **uniforme** sur les 3 langues. On évalue les métriques séparément pour le français, l'arabe et l'anglais.

In [ ]:
test_df = df.iloc[test_idx].copy()
test_df["prediction"] = y_pred

print("=" * 60)
print("📊 PERFORMANCE PAR LANGUE (sur le jeu de test)")
print("=" * 60)

for lang in sorted(test_df["language"].unique()):
    mask = test_df["language"] == lang
    if mask.sum() == 0:
        continue
    y_true_lang = test_df.loc[mask, "label"].values
    y_pred_lang = test_df.loc[mask, "prediction"].values
    
    acc = accuracy_score(y_true_lang, y_pred_lang)
    f1 = f1_score(y_true_lang, y_pred_lang, average="macro", zero_division=0)
    prec = precision_score(y_true_lang, y_pred_lang, zero_division=0)
    rec = recall_score(y_true_lang, y_pred_lang, zero_division=0)
    
    n_articles = test_df.loc[mask, "article_id"].nunique()
    print(f"\n🌍 {lang} ({mask.sum()} segments, {n_articles} articles)")
    print(f"   Accuracy  : {acc:.4f}")
    print(f"   Precision : {prec:.4f}")
    print(f"   Recall    : {rec:.4f}")
    print(f"   F1 macro  : {f1:.4f}")

### 8.4 Analyse des erreurs

On inspecte les segments mal classés pour comprendre les faiblesses du modèle.

In [ ]:
test_df["correct"] = test_df["label"] == test_df["prediction"]

# Faux Positifs : bruit classé comme éditorial
fp = test_df[(test_df["label"] == 0) & (test_df["prediction"] == 1)]
# Faux Négatifs : éditorial classé comme bruit
fn = test_df[(test_df["label"] == 1) & (test_df["prediction"] == 0)]

print(f"Erreurs totales : {(~test_df['correct']).sum()} / {len(test_df)}")
print(f"  • Faux Positifs (bruit → éditorial) : {len(fp)}")
print(f"  • Faux Négatifs (éditorial → bruit) : {len(fn)}")
print()

if len(fp) > 0:
    print("=" * 60)
    print("🔴 FAUX POSITIFS (bruit classé comme éditorial)")
    print("=" * 60)
    for _, row in fp.head(5).iterrows():
        print(f"  [{row['article_id']}] {row['segment'][:100]}")
        print()

if len(fn) > 0:
    print("=" * 60)
    print("🟡 FAUX NÉGATIFS (éditorial classé comme bruit)")
    print("=" * 60)
    for _, row in fn.head(5).iterrows():
        print(f"  [{row['article_id']}] {row['segment'][:100]}")
        print()